# DermAssist AI lab
A reproducible supervised-learning study, from a data audit to offline Android inference. Read `README.md` for the actual scope and limitations. This notebook is a companion to the executed scripts, not a claim of clinical accuracy.


## 1. Define the learning problem
Input: one RGB photo. Output: a pattern label among eczema, urticaria, and folliculitis, or abstention. Symptom notes are not predictor inputs. Explain why three supported classes do not mean all other diseases are absent.


In [ ]:
from pathlib import Path
import os, json, subprocess, sys
REPO = Path.cwd().parent if Path.cwd().name == "ml" else Path.cwd()
DATA = Path(os.environ.get("AI_DATA_DIR", str(REPO / "ml/data")))
RUN = DATA / "run-20261006"
os.environ["KERAS_HOME"] = str(DATA / "keras-cache")
os.environ["MPLCONFIGDIR"] = str(DATA / "matplotlib-cache")
print(REPO, DATA)


## 2. Audit, de-duplicate, and split
Before fitting any parameters, freeze train/tune/calibration/test cases. Examine class imbalance and missing skin-tone labels. Why would photos from the same case in different splits exaggerate generalization?


In [ ]:
subprocess.run([sys.executable, "ml/prepare_data.py", "--data-dir", str(DATA)], cwd=REPO, check=True)
audit = json.loads((DATA / "audit.json").read_text())
audit["split_counts"], audit["eligible_class_counts"]


## 3. Compare representations and learning algorithms
The same cases feed majority, logistic-regression, kNN, and MobileNet experiments. Standardization learns only from training data. Hyperparameters use tune macro-F1. What would a higher accuracy but lower macro-F1 suggest about minority classes?


## 4. Train a neural network
The scripts train a ReLU/dropout head on frozen ImageNet features, then fine-tune final convolution blocks with a lower Adam learning rate. Cross-entropy supplies gradients; class weights address imbalance; early stopping limits overfitting. The tune partition chooses the stage, not the test partition.


In [ ]:
# This cell performs real CPU training and overwrites the exported research model.
subprocess.run([sys.executable, "ml/train.py", "--data-dir", str(DATA), "--output", str(RUN)], cwd=REPO, check=True)


## 5. Interpret learning curves
Look for a growing gap between training and tune loss. Explain the bias/variance tradeoff and why the fine-tuned stage might lose to the simpler frozen stage.


In [ ]:
from IPython.display import display, Image
display(Image(filename=str(RUN / "learning_curves.png")))


## 6. Calibration and rejection
Temperature scaling minimizes calibration NLL. Joint score and cosine-distance thresholds can abstain. They are selected before final evaluation. Discuss why a high softmax score is not a patient’s probability of disease, and why OOD detection is imperfect.


In [ ]:
metrics = json.loads((RUN / "metrics.json").read_text())
metrics["selection"], metrics["selective"], metrics["ood"]


## 7. Hold-out evaluation and error analysis
Compare all baselines on the exact same held-out cases. Read per-class precision/recall/F1 and the confusion matrix. Distinguish empirical intervals from clinical validation. Do not tune the current model based on test errors; use a new version and an untouched future test set.


In [ ]:
display(Image(filename=str(RUN / "confusion_matrix.png")))
{k: {m: v[m] for m in ["accuracy", "macro_f1"]} for k,v in metrics["baselines"].items()}, metrics["neural"]


## 8. Fairness and robustness
Compare subgroup counts before comparing scores. A tiny or absent subgroup cannot support a fairness claim. Inspect blur/exposure sensitivity and recognize the difference between this US public dataset and the intended rural population.


In [ ]:
metrics["fairness"], metrics["robustness"]


## 9. Export and Android testing
Float16 weights reduce model size. Golden input/output vectors verify numerical consistency on Android. Device logs report measured emulator latency; physical-phone behavior is a separate test. Review `TESTING.md` and `ml/reports/` for executed evidence.


In [ ]:
metrics["export"], metrics["model_bytes"]


## 10. Viva / coursework discussion
1. Why split by case and remove duplicates before splitting?
2. Why compare against a majority predictor and classical learners?
3. Which parameters are learned by backpropagation?
4. How do early stopping, dropout, and class weighting change learning?
5. What does temperature scaling change, and what does it leave unchanged?
6. What happens to coverage when rejection thresholds tighten?
7. Why can an OOD sample receive a confident wrong prediction?
8. What evidence is missing before clinical use?
9. Which syllabus topics apply here, and which have not been implemented?
